# Storage durability checks — M2-1, M2-14, M3-5

Both cases are about `server/callbacks/utils/json_file_store.py` staying correct under conditions a normal
single-tester CLI session never exercises: two writers at once, and a crash mid-write. Both are naturally
suited to real-code testing — a human can't reliably time a `kill -9` to land mid-`write()`, but a notebook
can simulate the on-disk aftermath exactly and check the invariant directly.

**This notebook found a real bug while being built** (see the M2-14 section below) — kept in, not edited
out, because it's the clearest demonstration yet of why "test against the real code" beats "read the code
and trust the reasoning."

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import harness


repo root on sys.path: C:\Users\hp\Desktop\Aayush\repo


---
## M2-1 — two large concurrent writes must not interleave/corrupt the file

**Real-world scenario:** two large, genuinely simultaneous consent-notify POSTs land on the server at
close to the same instant (this server runs threaded via `asyncio.to_thread()` for every blocking file
write, so two in-flight requests really can call `_append()` on `consents.jsonl` concurrently). Python's
`open(path, "a")` gives no atomicity guarantee for a write larger than the OS pipe buffer — without a lock,
two big writes landing at the same moment could interleave their bytes mid-line, corrupting BOTH consent
records, not just delaying one of them.

**Fix:** a per-file `threading.Lock` in `_append()` serializes every write to a given file from within this
one process.

**Pass criteria:** after 20 large concurrent writes, every line in the file is still valid, unmangled JSON,
and all 20 records are independently correct.

In [2]:
import threading
import json as jsonmod

from server.callbacks.repository.consent_repository import save_consent, get_all_consents

scratch = harness.activate_scratch_storage("m2_1")

N = 20
def writer(i):
    save_consent(f"consent-concurrent-{i}", {
        "care_contexts": [{"referenceNumber": f"cc-{j}"} for j in range(30)],
        "padding": "x" * 500,  # inflate each line's size to make interleaving more likely if unlocked
    })

threads = [threading.Thread(target=writer, args=(i,)) for i in range(N)]
for t in threads:
    t.start()
for t in threads:
    t.join()

lines = (scratch / "consents.jsonl").read_text().splitlines()
all_parse = True
for ln in lines:
    try:
        jsonmod.loads(ln)
    except ValueError:
        all_parse = False

harness.check(f"all {len(lines)} lines are valid, unmangled JSON after {N} concurrent writers", all_parse and len(lines) == N)
harness.check("all 20 consents are independently retrievable and correct", len(get_all_consents()) == N)


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m2_1_vda1g932
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- all 20 lines are valid, unmangled JSON after 20 concurrent writers
PASS -- all 20 consents are independently retrievable and correct


True

---
## M2-14 — a crash mid-write must not corrupt the NEXT record too

**Original claim (Batch 2 runbook, "no fix written"):** "the append-only storage design means a crash
mid-write can only ever leave one partial trailing line, which `_replay()` already skips on read. There
should be no 'next record' to corrupt, since nothing is written after the in-flight line."

**What this notebook found:** that reasoning has a gap. `_append()`'s write is
`f.write(json.dumps(record) + "\n")` as one call — if the process is killed before the trailing `"\n"`
reaches disk, the file's last line has NO newline at the end. The very next `_append()` call (e.g. the
server's first write right after restarting) opens the file in append mode and writes straight onto the
end of that existing content — landing on the SAME physical line as the stale partial data, with nothing
separating them. `_replay()` reads that combined line, `json.loads()` fails on the whole thing, and the
fresh, otherwise-perfectly-good record written after restart is silently lost too — not just the one that
was genuinely in-flight during the crash.

**The fix** (applied directly in this session as a result of this finding, in
`server/callbacks/utils/json_file_store.py`): before writing, `_append()` now checks whether the file
already ends with a newline (or is empty/doesn't exist yet) and, if not, writes a leading `"\n"` first —
isolating any stale partial line onto its own (still skipped, but now harmless) line so a fresh write can
never be dragged into it.

**Pass criteria:** a fully-written record before the "crash" survives; the crash's own partial line is
skipped; a fresh write made right after (simulating a post-restart write) is intact and correctly
readable — separately from the corrupted line, not merged into it.

In [3]:
from server.callbacks.utils.json_file_store import set_key, get_key, get_all

scratch2 = harness.activate_scratch_storage("m2_14")

set_key("crash_test.jsonl", "good-record-1", {"value": "intact"})

# Simulate a kill -9 mid-write: append a truncated/garbled trailing line directly to the file
# (bypassing set_key, which always writes a complete, newline-terminated line) -- this is what a
# crash mid f.write(json.dumps(record) + "\n") can leave behind if the "\n" never made it to disk.
with open(scratch2 / "crash_test.jsonl", "a", encoding="utf-8") as f:
    f.write('{"key": "in-flight-record", "value": {"partial": "dat')  # deliberately cut off, NO trailing newline

try:
    state_after_crash = get_all("crash_test.jsonl")
    read_ok = True
except Exception:
    read_ok = False
    state_after_crash = {}

harness.check("read right after the simulated crash doesn't raise", read_ok)
harness.check("the earlier, fully-written record survives intact", state_after_crash.get("good-record-1") == {"value": "intact"})
harness.check("the partial/garbled in-flight line is skipped, not returned", "in-flight-record" not in state_after_crash)

# Now simulate the server restarting and making its first fresh write.
set_key("crash_test.jsonl", "fresh-record-after-restart", {"value": "post-restart-write"})

harness.check(
    "a fresh write right after the crash is stored and reads back correctly (THIS is the check that used to FAIL before the fix)",
    get_key("crash_test.jsonl", "fresh-record-after-restart") == {"value": "post-restart-write"},
)


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m2_14_e1v3d_gx
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- read right after the simulated crash doesn't raise
PASS -- the earlier, fully-written record survives intact
PASS -- the partial/garbled in-flight line is skipped, not returned
PASS -- a fresh write right after the crash is stored and reads back correctly (THIS is the check that used to FAIL before the fix)


True

---
## M3-5 — Concurrent writers to the pending Health Information Request store (private keys included)

**Real code under test:**
`server/callbacks/repository/pending_health_information_request_repository.py`, backed by the same
`json_file_store.py` module M2-1's fix already hardened.

**Real-world scenario:** M2-1 proved the underlying per-file lock protects `consents.jsonl` under
concurrent writers. This repository is a *different* file (`pending_health_information_requests.jsonl`),
used by a different flow (M3's HIU-initiated Health Information Request), and it stores something more
sensitive than a consent record -- each entry carries its own ECDH **private key** material generated for
that request. Worth independently verifying (per the tracker's own generic-solution rule: a shared fix
resolving multiple cases still gets each case verified on its own, never assumed) that the same generic
lock genuinely protects this file too, and that concurrent writes don't corrupt or cross-contaminate
private key material between requests.

**Pass criteria:** 20 concurrent writers, each saving its own request with its own private key → all 20
records read back intact, each with its own correct (not another request's) private key.

In [4]:
import threading

harness.activate_scratch_storage("m3_5")

from server.callbacks.repository.pending_health_information_request_repository import (
    save_pending_health_information_request, get_pending_health_information_request,
)

def write_worker(i):
    save_pending_health_information_request(f"req-m3-5-{i}", {
        "consent_id": f"consent-{i}",
        "key_material": {"private_key": f"PRIVATE_KEY_SECRET_{i}", "nonce": f"nonce-{i}"},
    })

threads = [threading.Thread(target=write_worker, args=(i,)) for i in range(20)]
[t.start() for t in threads]
[t.join() for t in threads]

all_ok = True
for i in range(20):
    rec = get_pending_health_information_request(f"req-m3-5-{i}")
    if rec is None or rec.get("key_material", {}).get("private_key") != f"PRIVATE_KEY_SECRET_{i}":
        all_ok = False
        print(f"BAD RECORD for req-m3-5-{i}: {rec}")

harness.check("20 concurrent writers (each carrying its own private key) -> all 20 records intact, no corruption/cross-contamination", all_ok)


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_5_p9_da4w0
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- 20 concurrent writers (each carrying its own private key) -> all 20 records intact, no corruption/cross-contamination


True

---
## M2-2 — a `delete` and a `save` landing at the same moment must not silently lose data (confirm-only)

**What this is**: `json_file_store.py`'s append-only design (already documented in its own module docstring, and already given a per-file `threading.Lock` for tracker case M2-1) means `set_key()` and `delete_key()` both go through the same serialized `_append()` call — whichever one's lock acquisition wins, its line lands in the file, and `_replay()` always uses "latest line for a key wins." There's no path where a concurrent set+delete corrupts the file or loses a write outright — the outcome is always one of the two operations' results, deterministically, never a torn/mixed state.

**No fix needed** — this cell races 40 concurrent `set_key()`/`delete_key()` calls against the exact same key and confirms: no exceptions, every appended line is still individually valid JSON (no interleaved/corrupted writes), and the final state is always one of the well-defined possible outcomes.

In [5]:
import threading
import json as _json

import server.callbacks.utils.json_file_store as jfs

harness.activate_scratch_storage("m2_2")

race_errors = []
def racer(i):
    try:
        if i % 2 == 0:
            jfs.set_key("race_test.jsonl", "shared-key", f"value-from-writer-{i}")
        else:
            jfs.delete_key("race_test.jsonl", "shared-key")
    except Exception as exc:
        race_errors.append(exc)

threads = [threading.Thread(target=racer, args=(i,)) for i in range(40)]
[t.start() for t in threads]
[t.join() for t in threads]

harness.check("40 concurrent set/delete calls on the same key -> no exceptions raised", len(race_errors) == 0)

final_value = jfs.get_key("race_test.jsonl", "shared-key")
harness.check(
    "final state is well-defined -- either a writer's value or deleted, never corrupted/unreadable",
    final_value is None or (isinstance(final_value, str) and final_value.startswith("value-from-writer-")),
)

log_lines = (jfs._STORAGE_ROOT / "race_test.jsonl").read_text(encoding="utf-8").splitlines()
all_valid = all(_json.loads(line) for line in log_lines if line.strip())
harness.check(f"all {len(log_lines)} appended lines are individually valid JSON (no interleaved/corrupted writes)", all_valid)


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m2_2_ce28qppm
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- 40 concurrent set/delete calls on the same key -> no exceptions raised
PASS -- final state is well-defined -- either a writer's value or deleted, never corrupted/unreadable
PASS -- all 40 appended lines are individually valid JSON (no interleaved/corrupted writes)


True

---
## M2-16 — a corrupted stored consent record (wrong `care_contexts` shape) must not silently drop the ABDM acknowledgment

**Real-world scenario**: a stored consent record's `care_contexts` field ends up in an unexpected shape — a string, a dict, an explicit `null`, or a list containing non-dict entries (a hand-edit, a future write-side bug, or storage corruption). The old `health_information_request_service.py` assumed `consent.get("care_contexts", [])` is always a list of dicts and iterated it directly — a corrupted shape raised an unhandled `AttributeError`/`TypeError`. That exception WAS caught by the function's outer try/except, so the server process didn't crash — but the catch happens AFTER the idempotency key is already marked processed and BEFORE the ABDM acknowledgment is ever sent, so the practical effect was a silently dropped request: ABDM gets no ack at all and just sees a timeout, with only a log line as evidence.

**Fix** (`server/callbacks/services/health_information_request_service.py`): the stored `care_contexts` field is now explicitly shape-checked before use — a non-list value is logged clearly and treated as zero approved care contexts (same "log clearly, don't crash" pattern as the M1-10/M1-12 guards) — letting processing continue far enough to still send ABDM a proper acknowledgment.

In [6]:
import asyncio
from unittest.mock import patch

import server.callbacks.services.health_information_request_service as hirs
from server.callbacks.repository.consent_repository import save_consent

harness.activate_scratch_storage("m2_16")

for bad_shape, label in [("not-a-list", "string"), ({"a": 1}, "dict"), (None, "explicit null"), (["not-a-dict"], "list of non-dicts")]:
    save_consent(f"consent-m2-16-{label}", {"hip_id": "HIP-1", "care_contexts": bad_shape})

    callback_data = {
        "headers": {"request-id": f"req-m2-16-{label}", "x-hip-id": "HIP-1"},
        "body": {
            "transactionId": f"txn-m2-16-{label}",
            "hiRequest": {
                "consent": {"id": f"consent-m2-16-{label}"},
                "dateRange": {"from": "2026-01-01", "to": "2026-01-02"},
                "dataPushUrl": "https://example.org/push",
                "keyMaterial": {},
            },
        },
    }

    ack_recorder = harness.CallRecorder(harness.FakeResponse(200, {}))
    # build_bundles_for_care_contexts() returning [] means the "no records
    # available" branch always fires below, which itself calls
    # send_health_information_notify() (a real outbound call) -- mocked too,
    # so this cell stays fully offline like every other harness-based test.
    with patch.object(hirs, "build_bundles_for_care_contexts", lambda refs, date_range=None: []), \
         patch.object(hirs, "send_health_information_notify", lambda **kw: harness.FakeResponse(202, {})), \
         patch.object(hirs, "send_on_health_information_request", ack_recorder):
        try:
            await hirs.process_health_information_request(callback_data)
            crashed_hirs = False
        except Exception as exc:
            crashed_hirs = True

    harness.check(f"care_contexts={label}: processed without crashing", not crashed_hirs)
    harness.check(f"care_contexts={label}: ABDM still got an acknowledgment (not silently dropped)", ack_recorder.call_count == 1)


2026-08-15 13:25:45  -> Data request received -- HIU wants the patient's records (POST /api/v3/hip/health-information/request)
2026-08-15 13:25:45  -> Extracted consent ID, approved care contexts, and encryption keys
2026-08-15 13:25:45     [ERROR] Stored consent consent-m2-16-string has a malformed 'care_contexts' field (expected a list, got str: 'not-a-list') -- treating as no approved care contexts rather than crashing.
2026-08-15 13:25:45  -> Assembled 0 FHIR record(s) for the approved care context(s)
2026-08-15 13:25:45     [API] Acknowledging Data Request to ABDM -- POST .../hip/on-request -> 200
2026-08-15 13:25:45     [API] Notifying ABDM of Transfer Outcome -- POST .../health-information/notify -> 202
2026-08-15 13:25:45     [ERROR] Could not prepare any FHIR records for the approved care contexts.
2026-08-15 13:25:45  -> Data request received -- HIU wants the patient's records (POST /api/v3/hip/health-information/request)
2026-08-15 13:25:45  -> Extracted consent ID, approved

[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m2_16_pgwi1_u6
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- care_contexts=string: processed without crashing
PASS -- care_contexts=string: ABDM still got an acknowledgment (not silently dropped)
PASS -- care_contexts=dict: processed without crashing
PASS -- care_contexts=dict: ABDM still got an acknowledgment (not silently dropped)
PASS -- care_contexts=explicit null: processed without crashing
PASS -- care_contexts=explicit null: ABDM still got an acknowledgment (not silently dropped)
PASS -- care_contexts=list of non-dicts: processed without crashing
PASS -- care_contexts=list of non-dicts: ABDM still got an acknowledgment (not silently dropped)
